# Session explorer

Loads one FIT file, runs the analysis pipeline (`analyze`) and plots the smoothed speed with the detected efforts.
Use it to calibrate `AnalysisParameters` on real files (milestone M0).

Before running: build the jars with `./gradlew :fit-parser:jar :analysis:jar` from the repository root.
Paths below are relative to this `notebooks/` directory.

In [ ]:
%use kandy, dataframe

In [ ]:
@file:DependsOn("com.garmin:fit:21.217.0")
@file:DependsOn("../domain/build/libs/domain-0.1.0-SNAPSHOT.jar")
@file:DependsOn("../fit-parser/build/libs/fit-parser-0.1.0-SNAPSHOT.jar")
@file:DependsOn("../analysis/build/libs/analysis-0.1.0-SNAPSHOT.jar")

In [ ]:
import com.ttaaa.ultimate.analysis.analyze
import com.ttaaa.ultimate.domain.AnalysisParameters
import com.ttaaa.ultimate.fit.FitParser
import java.io.File

val fitFile = File("../fit-parser/src/test/resources/fit/24557963847_ACTIVITY.fit")
val params = AnalysisParameters()
val raw = fitFile.inputStream().use { FitParser.parse(it) }
val result = analyze(raw, params)

"${result.recordingMode}: ${result.samples.size} samples, ${result.pauses.size} pauses, ${result.efforts.size} efforts"

In [ ]:
val series = dataFrameOf(
    "t" to result.samples.map { it.t },
    "speed" to result.samples.map { it.speed },
    "accel" to result.samples.map { it.accel },
    "inPause" to result.samples.map { it.inPause },
)
val efforts = dataFrameOf(
    "startT" to result.efforts.map { it.startT },
    "peakT" to result.efforts.map { it.peakT },
    "endT" to result.efforts.map { it.endT },
    "peakSpeed" to result.efforts.map { it.metrics.peakSpeed },
    "meanSpeedFirst3s" to result.efforts.map { it.metrics.meanSpeedFirst3s },
    "timeTo80PctPeakSec" to result.efforts.map { it.metrics.timeTo80PctPeakSec },
)
efforts

## Speed with detected efforts

Smoothed speed (m/s) over the session time `t` (s); red points mark the effort peaks.

In [ ]:
series.plot {
    line {
        x("t")
        y("speed")
    }
    withData(efforts) {
        points {
            x("peakT")
            y("peakSpeed")
            size = 4.0
            color = Color.RED
        }
    }
    layout {
        title = "Smoothed speed with detected efforts"
        size = 1200 to 400
    }
}

## Calibration

Change parameters and compare, e.g. a lower trigger acceleration:

In [ ]:
val tuned = analyze(raw, params.copy(effortStartAccel = 0.8))
"${tuned.efforts.size} efforts with effortStartAccel = 0.8 (default ${params.effortStartAccel}: ${result.efforts.size})"